In [2]:
from collections import deque

def ac3(domains, neighbors):
    """
    AC-3 algorithm for constraint propagation.
    """
    queue = deque()

    # Initialize queue with all arcs
    for var in neighbors:
        for neighbor in neighbors[var]:
            queue.append((var, neighbor))

    while queue:
        xi, xj = queue.popleft()

        if remove_inconsistent_values(xi, xj, domains):
            if len(domains[xi]) == 0:
                return False

            for xk in neighbors[xi]:
                if xk != xj:
                    queue.append((xk, xi))

    return True


def remove_inconsistent_values(xi, xj, domains):
    """
    Remove values from xi's domain that have no compatible value
    in xj's domain.
    """
    removed = False

    for x in set(domains[xi]):
        # Adjacent regions must have different colors
        if all(x == y for y in domains[xj]):
            domains[xi].remove(x)
            removed = True

    return removed


def backtrack(assignment, domains, neighbors):
    """
    Backtracking search using the MRV heuristic.
    """
    # If all variables are assigned, return the assignment
    if len(assignment) == len(domains):
        return assignment

    # Select the unassigned variable with the smallest domain
    unassigned_vars = [
        v for v in domains if v not in assignment
    ]

    var = min(unassigned_vars, key=lambda v: len(domains[v]))

    for value in domains[var]:
        # Check consistency with assigned neighbors
        if all(
            assignment.get(neigh) != value
            for neigh in neighbors[var]
        ):
            assignment[var] = value

            # Copy domains for constraint propagation
            local_domains = {
                v: list(domains[v]) for v in domains
            }

            local_domains[var] = [value]

            # Apply AC-3 constraint propagation
            if ac3(local_domains, neighbors):
                result = backtrack(
                    assignment, local_domains, neighbors
                )

                if result:
                    return result

            # Backtrack if no solution is found
            assignment.pop(var)

    return None


def main():
    # Variables representing Australian regions
    variables = ['WA', 'NT', 'SA', 'Q', 'NSW', 'V', 'T']

    # Available colors
    colors = ['Red', 'Green', 'Blue']

    # Initialize domains
    domains = {
        var: colors[:] for var in variables
    }

    # Neighboring regions
    neighbors = {
        'WA': ['NT', 'SA'],
        'NT': ['WA', 'SA', 'Q'],
        'SA': ['WA', 'NT', 'Q', 'NSW', 'V'],
        'Q': ['NT', 'SA', 'NSW'],
        'NSW': ['Q', 'SA', 'V'],
        'V': ['SA', 'NSW'],
        'T': []
    }

    assignment = {}

    solution = backtrack(
        assignment, domains, neighbors
    )

    if solution:
        print("Solution found:")
        for region in sorted(solution):
            print(f"{region}: {solution[region]}")
    else:
        print("No solution found.")


if __name__ == "__main__":
    main()

Solution found:
NSW: Green
NT: Green
Q: Red
SA: Blue
T: Red
V: Red
WA: Red
